In [ ]:
# ============================================================
# CELL 1
# DISTILBERT POLITICAL BIAS CLASSIFICATION
# DATA PREPROCESSING + BALANCING + TOKENIZATION
# ============================================================

# ============================================================
# INSTALLS
# ============================================================

# Uncomment if needed

# !pip install -q transformers datasets accelerate scikit-learn seaborn

# ============================================================
# IMPORTS
# ============================================================

import os
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import resample

import torch

from datasets import Dataset

from transformers import DistilBertTokenizerFast

# ============================================================
# SEED
# ============================================================

def seed_everything(seed=42):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(seed)

seed_everything(42)

# ============================================================
# DEVICE
# ============================================================

device = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"
)

print("="*100)
print("DEVICE")
print("="*100)

print(f"\nUsing Device: {device}")

# ============================================================
# LOAD DATASET
# ============================================================

from pathlib import Path

# Finds the attached Kaggle dataset by file name, including ponssvvv/bias-clean.
_dataset_files = sorted(Path("/kaggle/input").rglob("bias_clean.csv"))
if not _dataset_files:
    raise FileNotFoundError("Attach the bias_clean Kaggle dataset (ponssvvv/bias-clean) in the notebook Input panel.")
DATA_PATH = str(_dataset_files[0])
print(f"Using dataset: {DATA_PATH}")

df = pd.read_csv(

    DATA_PATH
)

# ============================================================
# CLEAN DATA
# ============================================================

df = df.dropna(

    subset=["page_text", "bias"]
)

df = df.reset_index(drop=True)

# ============================================================
# BASIC CLEANING
# ============================================================

def clean_text(text):

    text = str(text)

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"\s+", " ", text)

    text = text.strip()

    return text

df["page_text"] = df["page_text"].apply(

    clean_text
)

# ============================================================
# LABEL DISTRIBUTION BEFORE BALANCING
# ============================================================

print("\n" + "="*100)
print("ORIGINAL LABEL DISTRIBUTION")
print("="*100)

print(df["bias"].value_counts())

# ============================================================
# BALANCE CLASSES
# ============================================================

min_class_size = df["bias"].value_counts().min()

balanced_df = pd.DataFrame()

for label in df["bias"].unique():

    subset = df[df["bias"] == label]

    subset = resample(

        subset,

        replace=False,

        n_samples=min_class_size,

        random_state=42
    )

    balanced_df = pd.concat(

        [balanced_df, subset]
    )

balanced_df = balanced_df.sample(

    frac=1,

    random_state=42
).reset_index(drop=True)

# ============================================================
# LABEL DISTRIBUTION AFTER BALANCING
# ============================================================

print("\n" + "="*100)
print("BALANCED LABEL DISTRIBUTION")
print("="*100)

print(balanced_df["bias"].value_counts())

# ============================================================
# LABEL ENCODING
# ============================================================

label_encoder = LabelEncoder()

balanced_df["label"] = label_encoder.fit_transform(

    balanced_df["bias"]
)

# ============================================================
# SAVE LABEL MAPPING
# ============================================================

label_mapping = {

    idx: label

    for idx, label in enumerate(

        label_encoder.classes_
    )
}

print("\n" + "="*100)
print("LABEL MAPPING")
print("="*100)

print(label_mapping)

# ============================================================
# TRAIN / VALID / TEST SPLIT
# ============================================================

train_df, temp_df = train_test_split(

    balanced_df,

    test_size=0.30,

    stratify=balanced_df["label"],

    random_state=42
)

valid_df, test_df = train_test_split(

    temp_df,

    test_size=0.50,

    stratify=temp_df["label"],

    random_state=42
)

print("\n" + "="*100)
print("DATASET SPLITS")
print("="*100)

print(f"\nTrain Size      : {len(train_df)}")
print(f"Validation Size : {len(valid_df)}")
print(f"Test Size       : {len(test_df)}")

# ============================================================
# TOKENIZER
# ============================================================

MODEL_NAME = "distilbert-base-uncased"

tokenizer = DistilBertTokenizerFast.from_pretrained(

    MODEL_NAME
)

MAX_LENGTH = 256

# ============================================================
# TOKENIZATION FUNCTION
# ============================================================

def tokenize_function(examples):

    return tokenizer(

        examples["page_text"],

        truncation=True,

        padding="max_length",

        max_length=MAX_LENGTH
    )

# ============================================================
# CONVERT TO HF DATASETS
# ============================================================

train_dataset = Dataset.from_pandas(

    train_df[["page_text", "label"]]
)

valid_dataset = Dataset.from_pandas(

    valid_df[["page_text", "label"]]
)

test_dataset = Dataset.from_pandas(

    test_df[["page_text", "label"]]
)

# ============================================================
# TOKENIZE
# ============================================================

train_dataset = train_dataset.map(

    tokenize_function,

    batched=True
)

valid_dataset = valid_dataset.map(

    tokenize_function,

    batched=True
)

test_dataset = test_dataset.map(

    tokenize_function,

    batched=True
)

# ============================================================
# FORMAT DATASETS
# ============================================================

train_dataset.set_format(

    type="torch",

    columns=[

        "input_ids",

        "attention_mask",

        "label"
    ]
)

valid_dataset.set_format(

    type="torch",

    columns=[

        "input_ids",

        "attention_mask",

        "label"
    ]
)

test_dataset.set_format(

    type="torch",

    columns=[

        "input_ids",

        "attention_mask",

        "label"
    ]
)

# ============================================================
# LABEL DISTRIBUTION PLOT
# ============================================================

plt.figure(figsize=(10,6))

sns.countplot(

    x=balanced_df["bias"]
)

plt.title("Balanced Political Bias Distribution")

plt.xticks(rotation=20)

plt.show()

# ============================================================
# TEXT LENGTH DISTRIBUTION
# ============================================================

balanced_df["text_length"] = balanced_df["page_text"].apply(

    lambda x: len(str(x).split())
)

plt.figure(figsize=(10,6))

sns.histplot(

    balanced_df["text_length"],

    bins=50
)

plt.title("Text Length Distribution")

plt.xlabel("Word Count")

plt.show()

# ============================================================
# SAVE DATASETS
# ============================================================

train_df.to_csv(

    "distilbert_train.csv",

    index=False
)

valid_df.to_csv(

    "distilbert_valid.csv",

    index=False
)

test_df.to_csv(

    "distilbert_test.csv",

    index=False
)

print("\n" + "="*100)
print("CELL 1 COMPLETED SUCCESSFULLY")
print("="*100)

In [ ]:
# ============================================================
# CELL 2
# DISTILBERT TRAINING + ADVANCED EVALUATION
# 40 EPOCHS + PATIENCE 4
# OLD TRANSFORMERS COMPATIBLE VERSION
# ============================================================

# ============================================================
# IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from sklearn.metrics import (

    accuracy_score,

    precision_score,

    recall_score,

    f1_score,

    confusion_matrix,

    classification_report
)

from transformers import (

    DistilBertForSequenceClassification,

    TrainingArguments,

    Trainer,

    EarlyStoppingCallback
)

# ============================================================
# LOAD MODEL
# ============================================================

print("="*100)
print("LOADING DISTILBERT MODEL")
print("="*100)

model = DistilBertForSequenceClassification.from_pretrained(

    MODEL_NAME,

    num_labels=len(label_mapping)
)

model.to(device)

print("\nModel Loaded Successfully.")

# ============================================================
# METRICS FUNCTION
# ============================================================

def compute_metrics(pred):

    labels = pred.label_ids

    preds = pred.predictions.argmax(-1)

    accuracy = accuracy_score(

        labels,

        preds
    )

    precision = precision_score(

        labels,

        preds,

        average="weighted"
    )

    recall = recall_score(

        labels,

        preds,

        average="weighted"
    )

    f1 = f1_score(

        labels,

        preds,

        average="weighted"
    )

    return {

        "accuracy": accuracy,

        "precision": precision,

        "recall": recall,

        "f1": f1
    }

# ============================================================
# TRAINING ARGUMENTS
# ============================================================

training_args = TrainingArguments(

    output_dir="./distilbert_results",

    num_train_epochs=40,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    warmup_steps=100,

    weight_decay=0.01,

    logging_steps=20,

    save_steps=100,

    eval_steps=100,

    save_strategy="steps",

    eval_strategy="steps",

    do_train=True,

    do_eval=True,

    fp16=torch.cuda.is_available(),

    load_best_model_at_end=True,

    metric_for_best_model="f1"
)

# ============================================================
# TRAINER
# ============================================================

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=valid_dataset,

    compute_metrics=compute_metrics,

    callbacks=[

        EarlyStoppingCallback(

            early_stopping_patience=4
        )
    ]
)

# ============================================================
# START TRAINING
# ============================================================

print("\n" + "="*100)
print("STARTING DISTILBERT TRAINING")
print("="*100)

trainer.train()

# ============================================================
# SAVE MODEL
# ============================================================

print("\n" + "="*100)
print("SAVING MODEL")
print("="*100)

trainer.save_model(

    "final_distilbert_bias_model"
)

tokenizer.save_pretrained(

    "final_distilbert_bias_model"
)

print("\nModel Saved Successfully.")

# ============================================================
# TEST EVALUATION
# ============================================================

print("\n" + "="*100)
print("TEST EVALUATION")
print("="*100)

predictions = trainer.predict(

    test_dataset
)

preds = np.argmax(

    predictions.predictions,

    axis=1
)

labels = predictions.label_ids

# ============================================================
# ADVANCED METRICS
# ============================================================

accuracy = accuracy_score(

    labels,

    preds
)

precision = precision_score(

    labels,

    preds,

    average="weighted"
)

recall = recall_score(

    labels,

    preds,

    average="weighted"
)

f1 = f1_score(

    labels,

    preds,

    average="weighted"
)

# ============================================================
# PRINT METRICS
# ============================================================

print("\n" + "="*100)
print("ADVANCED EVALUATION METRICS")
print("="*100)

print(f"\nAccuracy  : {accuracy:.4f}")

print(f"Precision : {precision:.4f}")

print(f"Recall    : {recall:.4f}")

print(f"F1-Score  : {f1:.4f}")

# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print("\n" + "="*100)
print("CLASSIFICATION REPORT")
print("="*100)

print(

    classification_report(

        labels,

        preds,

        target_names=list(label_mapping.values())
    )
)

# ============================================================
# CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(

    labels,

    preds
)

plt.figure(figsize=(10,8))

sns.heatmap(

    cm,

    annot=True,

    fmt="d",

    cmap="Blues",

    xticklabels=label_mapping.values(),

    yticklabels=label_mapping.values()
)

plt.title("DistilBERT Confusion Matrix")

plt.xlabel("Predicted")

plt.ylabel("Actual")

plt.show()

# ============================================================
# METRIC BAR PLOT
# ============================================================

metrics = [

    "Accuracy",

    "Precision",

    "Recall",

    "F1"
]

scores = [

    accuracy,

    precision,

    recall,

    f1
]

plt.figure(figsize=(8,5))

plt.bar(

    metrics,

    scores
)

plt.ylim(0,1)

plt.title("DistilBERT Evaluation Metrics")

plt.ylabel("Score")

plt.grid(axis='y')

plt.show()

# ============================================================
# PREDICTION DISTRIBUTION
# ============================================================

predicted_labels = [

    label_mapping[p]

    for p in preds
]

plt.figure(figsize=(10,6))

sns.countplot(

    x=predicted_labels
)

plt.title("Prediction Distribution")

plt.xticks(rotation=20)

plt.show()

# ============================================================
# FINAL SUMMARY TABLE
# ============================================================

results_df = pd.DataFrame({

    "Metric": [

        "Accuracy",

        "Precision",

        "Recall",

        "F1-Score"
    ],

    "Score": [

        accuracy,

        precision,

        recall,

        f1
    ]
})

print("\n" + "="*100)
print("FINAL METRIC SUMMARY")
print("="*100)

print(results_df)

# ============================================================
# FINAL SUCCESS
# ============================================================

print("\n" + "="*100)
print("CELL 2 COMPLETED SUCCESSFULLY")
print("="*100)

print("""

COMPLETED:

✓ DistilBERT Fine-Tuning
✓ 40 Epoch Training
✓ Early Stopping (Patience 4)
✓ Accuracy Analysis
✓ Precision Analysis
✓ Recall Analysis
✓ F1-Score Analysis
✓ Confusion Matrix
✓ Classification Report
✓ Metric Visualization
✓ Prediction Distribution Analysis

NEXT:

CELL 3 → SHAP XAI + Explainability

""")

In [ ]:
# ============================================================
# CELL 3
# FINAL ADVANCED EVALUATION
# DISTILBERT POLITICAL BIAS CLASSIFICATION
# NO XAI VERSION
# ============================================================

# ============================================================
# IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from sklearn.metrics import (

    accuracy_score,

    precision_score,

    recall_score,

    f1_score,

    matthews_corrcoef,

    cohen_kappa_score,

    confusion_matrix,

    classification_report
)

from transformers import (

    DistilBertTokenizerFast,

    DistilBertForSequenceClassification,

    pipeline
)

# ============================================================
# DEVICE
# ============================================================

device = 0 if torch.cuda.is_available() else -1

print("="*100)
print("DEVICE")
print("="*100)

print(f"\nUsing Device: {device}")

# ============================================================
# LOAD SAVED MODEL
# ============================================================

MODEL_PATH = "final_distilbert_bias_model"

print("\n" + "="*100)
print("LOADING SAVED DISTILBERT MODEL")
print("="*100)

tokenizer = DistilBertTokenizerFast.from_pretrained(

    MODEL_PATH
)

model = DistilBertForSequenceClassification.from_pretrained(

    MODEL_PATH
)

print("\nModel Loaded Successfully.")

# ============================================================
# CREATE PIPELINE
# ============================================================

classifier = pipeline(

    "text-classification",

    model=model,

    tokenizer=tokenizer,

    device=device,

    truncation=True,

    max_length=512
)

print("\nPipeline Created Successfully.")

# ============================================================
# LOAD TEST DATA
# ============================================================

test_df = pd.read_csv(

    "distilbert_test.csv"
)

print("\n" + "="*100)
print("TEST DATA LOADED")
print("="*100)

print(f"\nTest Samples: {len(test_df)}")

# ============================================================
# LABEL MAPPING
# ============================================================

id2label = {

    i: label

    for i, label in enumerate(

        sorted(test_df["bias"].unique())
    )
}

# ============================================================
# FULL TEST PREDICTIONS
# ============================================================

print("\n" + "="*100)
print("RUNNING COMPLETE TEST EVALUATION")
print("="*100)

texts = test_df["page_text"].tolist()

true_labels = test_df["label"].tolist()

predicted_labels = []

confidence_scores = []

# ============================================================
# INFERENCE LOOP
# ============================================================

for text in texts:

    pred = classifier(

        text,

        truncation=True,

        max_length=512
    )

    predicted_label = int(

        pred[0]["label"].split("_")[-1]
    )

    predicted_labels.append(

        predicted_label
    )

    confidence_scores.append(

        pred[0]["score"]
    )

# ============================================================
# ADVANCED METRICS
# ============================================================

accuracy = accuracy_score(

    true_labels,

    predicted_labels
)

precision = precision_score(

    true_labels,

    predicted_labels,

    average="weighted"
)

recall = recall_score(

    true_labels,

    predicted_labels,

    average="weighted"
)

weighted_f1 = f1_score(

    true_labels,

    predicted_labels,

    average="weighted"
)

macro_f1 = f1_score(

    true_labels,

    predicted_labels,

    average="macro"
)

mcc = matthews_corrcoef(

    true_labels,

    predicted_labels
)

kappa = cohen_kappa_score(

    true_labels,

    predicted_labels
)

# ============================================================
# PRINT METRICS
# ============================================================

print("\n" + "="*100)
print("ADVANCED METRICS")
print("="*100)

print(f"\nAccuracy           : {accuracy:.4f}")

print(f"Precision          : {precision:.4f}")

print(f"Recall             : {recall:.4f}")

print(f"Weighted F1        : {weighted_f1:.4f}")

print(f"Macro F1           : {macro_f1:.4f}")

print(f"Matthews Corrcoef  : {mcc:.4f}")

print(f"Cohen Kappa        : {kappa:.4f}")

# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print("\n" + "="*100)
print("CLASSIFICATION REPORT")
print("="*100)

print(

    classification_report(

        true_labels,

        predicted_labels
    )
)

# ============================================================
# CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(

    true_labels,

    predicted_labels
)

plt.figure(figsize=(10,8))

sns.heatmap(

    cm,

    annot=True,

    fmt="d",

    cmap="Blues"
)

plt.title("DistilBERT Confusion Matrix")

plt.xlabel("Predicted")

plt.ylabel("Actual")

plt.show()

# ============================================================
# ADVANCED METRIC BAR PLOT
# ============================================================

metric_names = [

    "Accuracy",

    "Precision",

    "Recall",

    "Weighted F1",

    "Macro F1",

    "MCC",

    "Kappa"
]

metric_scores = [

    accuracy,

    precision,

    recall,

    weighted_f1,

    macro_f1,

    mcc,

    kappa
]

plt.figure(figsize=(12,6))

plt.bar(

    metric_names,

    metric_scores
)

plt.title("Advanced DistilBERT Metrics")

plt.ylabel("Score")

plt.ylim(0,1)

plt.grid(axis='y')

plt.show()

# ============================================================
# CONFIDENCE SCORE DISTRIBUTION
# ============================================================

plt.figure(figsize=(10,6))

sns.histplot(

    confidence_scores,

    bins=30
)

plt.title("Prediction Confidence Distribution")

plt.xlabel("Confidence Score")

plt.ylabel("Frequency")

plt.show()

# ============================================================
# PREDICTION DISTRIBUTION
# ============================================================

predicted_names = [

    id2label[p]

    for p in predicted_labels
]

plt.figure(figsize=(10,6))

sns.countplot(

    x=predicted_names
)

plt.title("Prediction Distribution")

plt.xticks(rotation=20)

plt.show()

# ============================================================
# PER-CLASS ACCURACY
# ============================================================

per_class_accuracy = {}

for class_id in np.unique(true_labels):

    idxs = np.where(

        np.array(true_labels) == class_id
    )[0]

    class_acc = accuracy_score(

        np.array(true_labels)[idxs],

        np.array(predicted_labels)[idxs]
    )

    per_class_accuracy[

        id2label[class_id]
    ] = class_acc

# ============================================================
# PER-CLASS ACCURACY PLOT
# ============================================================

plt.figure(figsize=(10,6))

plt.bar(

    per_class_accuracy.keys(),

    per_class_accuracy.values()
)

plt.title("Per-Class Accuracy")

plt.ylabel("Accuracy")

plt.ylim(0,1)

plt.xticks(rotation=20)

plt.grid(axis='y')

plt.show()

# ============================================================
# FINAL SUMMARY TABLE
# ============================================================

results_df = pd.DataFrame({

    "Metric": [

        "Accuracy",

        "Precision",

        "Recall",

        "Weighted F1",

        "Macro F1",

        "MCC",

        "Cohen Kappa"
    ],

    "Score": [

        accuracy,

        precision,

        recall,

        weighted_f1,

        macro_f1,

        mcc,

        kappa
    ]
})

print("\n" + "="*100)
print("FINAL SUMMARY")
print("="*100)

print(results_df)

# ============================================================
# FINAL SUCCESS
# ============================================================

print("\n" + "="*100)
print("CELL 3 COMPLETED SUCCESSFULLY")
print("="*100)

print("""

COMPLETED:

✓ Advanced Accuracy Analysis
✓ Precision Analysis
✓ Recall Analysis
✓ Weighted F1
✓ Macro F1
✓ Matthews Correlation Coefficient
✓ Cohen Kappa
✓ Classification Report
✓ Confusion Matrix
✓ Confidence Distribution
✓ Prediction Distribution
✓ Per-Class Accuracy Analysis
✓ Advanced Visualization

FINAL PIPELINE:

DistilBERT Political Bias Classification System

""")

In [ ]:
# Reproducibility export: metrics and evaluation artifacts for the Kaggle output bundle.
from pathlib import Path
import json
import pandas as pd

_output_dir = Path("/kaggle/working/results")
_output_dir.mkdir(parents=True, exist_ok=True)
_metrics = {
    "model": 'DistilBERT',
    "accuracy": float(accuracy),
    "precision": float(precision),
    "recall": float(recall),
    "weighted_f1": float(weighted_f1),
    "macro_f1": float(macro_f1),
    "matthews_corrcoef": float(mcc),
    "cohen_kappa": float(kappa),
    "test_samples": int(len(test_df)),
}
(_output_dir / "distilbert_metrics.json").write_text(json.dumps(_metrics, indent=2), encoding="utf-8")
if "cm" in globals():
    pd.DataFrame(cm).to_csv(_output_dir / "distilbert_confusion_matrix.csv", index=False)
if "labels" in globals() and "preds" in globals():
    from sklearn.metrics import classification_report
    _names = list(label_mapping.values()) if "label_mapping" in globals() else None
    _report = classification_report(labels, preds, target_names=_names, output_dict=True, zero_division=0)
    pd.DataFrame(_report).transpose().to_csv(_output_dir / "distilbert_classification_report.csv")
print("Saved metrics and evaluation artifacts under /kaggle/working/results")